In [1]:
print("Page Replacement Simulator")

Page Replacement Simulator


 PART A:
 FIRST IN FIRST OUT

In [6]:
pages = [1, 2, 3, 1, 4, 5]
frames = 3
memory = []
page_faults = 0
page_hits = 0
print("Pages:", pages)
print("Number of Frames:", frames)

for page in pages:
    if page not in memory:
      page_faults += 1
      if len(memory) < frames:
        memory.append(page)
      else:
        memory.pop(0)
        memory.append(page)

    else:
        page_hits += 1


    print("page:",page, "Memory:", memory)

print("Page Faults:", page_faults)
print("Page Hits:", page_hits)

Pages: [1, 2, 3, 1, 4, 5]
Number of Frames: 3
page: 1 Memory: [1]
page: 2 Memory: [1, 2]
page: 3 Memory: [1, 2, 3]
page: 1 Memory: [1, 2, 3]
page: 4 Memory: [2, 3, 4]
page: 5 Memory: [3, 4, 5]
Page Faults: 5
Page Hits: 1


LRU ALGORITHM


In [8]:
pages = [1, 2, 3, 1, 4, 5]
frames = 3
memory = []
page_faults = 0
page_hits = 0

for page in pages:
    if page  in memory:
      page_hits += 1
      memory.remove(page)
      memory.append(page)
    else:
      page_faults += 1
      if len(memory) < frames:
        memory.append(page)
      else:
        memory.pop(0)
        memory.append(page)

    print("page:",page, "Memory:", memory)

print("Page Faults:", page_faults)
print("Page Hits:", page_hits)

page: 1 Memory: [1]
page: 2 Memory: [1, 2]
page: 3 Memory: [1, 2, 3]
page: 1 Memory: [2, 3, 1]
page: 4 Memory: [3, 1, 4]
page: 5 Memory: [1, 4, 5]
Page Faults: 5
Page Hits: 1


OPTIMAL ALGORITHM


In [10]:
pages = [1, 2, 3, 1, 4, 5]
frames = 3
memory = []
page_faults = 0
page_hits = 0

for i in range(len(pages)):
  page = pages[i]
  if page in memory:
    page_hits += 1
  else:
    page_faults += 1
    if len(memory) < frames:
      memory.append(page)
    else:
      future = pages[i+1:]
      distances = []

      for p in memory:
        if p in future:
          distances.append(future.index(p))
        else:
          distances.append(float('inf'))

      replace_index = distances.index(max(distances))
      memory[replace_index] = page

    print("page:",page, "Memory:", memory)

print("Page Faults:", page_faults)
print("Page Hits:", page_hits)

page: 1 Memory: [1]
page: 2 Memory: [1, 2]
page: 3 Memory: [1, 2, 3]
page: 4 Memory: [4, 2, 3]
page: 5 Memory: [5, 2, 3]
Page Faults: 5
Page Hits: 1


In [15]:


# Operating Systems Lab Assignment
# Page Replacement Algorithm Simulator
# Algorithms: FIFO, LRU and Optimal

def fifo(reference, num_frames):
    frames = []
    page_faults = 0
    page_hits = 0
    trace = []

    for step, page in enumerate(reference, start=1):
        replaced = "-"

        if page in frames:
            page_hits += 1
            status = "HIT"
        else:
            page_faults += 1
            status = "FAULT"

            if len(frames) < num_frames:
                frames.append(page)
            else:
                replaced = frames.pop(0)
                frames.append(page)

        trace.append({
            "step": step,
            "page": page,
            "frames": frames.copy(),
            "status": status,
            "replaced": replaced
        })

    return page_faults, page_hits, trace


def lru(reference, num_frames):
    frames = []
    page_faults = 0
    page_hits = 0
    trace = []

    for step, page in enumerate(reference, start=1):
        replaced = "-"

        if page in frames:
            page_hits += 1
            status = "HIT"

            # Move the recently used page to the end
            frames.remove(page)
            frames.append(page)
        else:
            page_faults += 1
            status = "FAULT"

            if len(frames) < num_frames:
                frames.append(page)
            else:
                replaced = frames.pop(0)
                frames.append(page)

        trace.append({
            "step": step,
            "page": page,
            "frames": frames.copy(),
            "status": status,
            "replaced": replaced
        })

    return page_faults, page_hits, trace


def optimal(reference, num_frames):
    frames = []
    page_faults = 0
    page_hits = 0
    trace = []

    for i, page in enumerate(reference):
        step = i + 1
        replaced = "-"

        if page in frames:
            page_hits += 1
            status = "HIT"
        else:
            page_faults += 1
            status = "FAULT"

            if len(frames) < num_frames:
                frames.append(page)
            else:
                future = reference[i + 1:]
                next_use = []

                for old_page in frames:
                    if old_page in future:
                        next_use.append(future.index(old_page))
                    else:
                        next_use.append(float("inf"))

                # Replace the page used farthest in the future
                # In a tie, replace the first matching frame
                replace_index = next_use.index(max(next_use))
                replaced = frames[replace_index]
                frames[replace_index] = page

        trace.append({
            "step": step,
            "page": page,
            "frames": frames.copy(),
            "status": status,
            "replaced": replaced
        })

    return page_faults, page_hits, trace


def display_trace(name, result, num_frames, total_pages):
    faults, hits, trace = result

    print("\n" + "=" * 76)
    print(f"{name} PAGE REPLACEMENT")
    print("=" * 76)

    header = f"{'Step':<7}{'Page':<7}"
    for i in range(num_frames):
        header += f"{('Frame ' + str(i + 1)):<10}"

    header += f"{'Status':<10}{'Replaced':<10}"
    print(header)
    print("-" * len(header))

    for item in trace:
        frame_values = item["frames"] + ["-"] * (
            num_frames - len(item["frames"])
        )

        row = f"{item['step']:<7}{item['page']:<7}"

        for value in frame_values:
            row += f"{str(value):<10}"

        row += f"{item['status']:<10}{str(item['replaced']):<10}"
        print(row)

    hit_ratio = hits / total_pages
    fault_ratio = faults / total_pages

    print("-" * 50)
    print("Total Page Faults :", faults)
    print("Total Page Hits   :", hits)
    print(f"Hit Ratio         : {hit_ratio:.2%}")
    print(f"Fault Ratio       : {fault_ratio:.2%}")

    return faults, hits, hit_ratio, fault_ratio


def main():
    print("=" * 50)
    print("PAGE REPLACEMENT ALGORITHM SIMULATOR")
    print("Algorithms: FIFO, LRU and Optimal")
    print("=" * 50)

    try:
        raw = input(
            "\nEnter page reference string (space separated): "
        ).strip()

        if not raw:
            print("Error: Reference string cannot be empty.")
            return

        reference = list(map(int, raw.split()))

        raw_frames = input("Enter number of frames: ").strip()
        num_frames = int(raw_frames)

        if num_frames < 1:
            print("Error: Number of frames must be at least 1.")
            return

        if any(page < 0 for page in reference):
            print("Error: Page numbers cannot be negative.")
            return

    except ValueError:
        print("Error: Enter only valid integers.")
        return

    print("\nReference String:", reference)
    print("Number of Frames:", num_frames)

    results = {
        "FIFO": fifo(reference, num_frames),
        "LRU": lru(reference, num_frames),
        "Optimal": optimal(reference, num_frames)
    }

    summary = {}

    # Display each algorithm's trace
    for name, result in results.items():
        summary[name] = display_trace(
            name, result, num_frames, len(reference)
        )

    # Final comparison
    print("\n" + "=" * 70)
    print("FINAL PERFORMANCE COMPARISON")
    print("=" * 70)

    print(
        f"{'Algorithm':<12}{'Faults':<10}{'Hits':<10}"
        f"{'Hit Ratio':<14}{'Fault Ratio':<14}"
    )
    print("-" * 60)

    for name, values in summary.items():
        faults, hits, hit_ratio, fault_ratio = values

        print(
            f"{name:<12}{faults:<10}{hits:<10}"
            f"{hit_ratio:<14.2%}{fault_ratio:<14.2%}"
        )

    best = min(summary, key=lambda name: summary[name][0])

    print("-" * 60)
    print("Fewest page faults:", best)
    print("Note: Optimal is the theoretical benchmark algorithm.")


main()



PAGE REPLACEMENT ALGORITHM SIMULATOR
Algorithms: FIFO, LRU and Optimal

Enter page reference string (space separated): 1 2 3 1 4 2 5
Enter number of frames: 3

Reference String: [1, 2, 3, 1, 4, 2, 5]
Number of Frames: 3

FIFO PAGE REPLACEMENT
Step   Page   Frame 1   Frame 2   Frame 3   Status    Replaced  
----------------------------------------------------------------
1      1      1         -         -         FAULT     -         
2      2      1         2         -         FAULT     -         
3      3      1         2         3         FAULT     -         
4      1      1         2         3         HIT       -         
5      4      2         3         4         FAULT     1         
6      2      2         3         4         HIT       -         
7      5      3         4         5         FAULT     2         
--------------------------------------------------
Total Page Faults : 5
Total Page Hits   : 2
Hit Ratio         : 28.57%
Fault Ratio       : 71.43%

LRU PAGE REPLACEMENT
S

## Part B: Manual Tracing of Page Replacement Algorithms

**Reference String:** 7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2  
**Number of Frames:** 3

### 1. FIFO (First In First Out)

| Page | 7 | 0 | 1 | 2 | 0 | 3 | 0 | 4 | 2 | 3 | 0 | 3 | 2 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Frame 1 | 7 | 7 | 7 | 2 | 2 | 2 | 2 | 4 | 4 | 4 | 0 | 0 | 0 |
| Frame 2 | - | 0 | 0 | 0 | 0 | 3 | 3 | 3 | 2 | 2 | 2 | 2 | 2 |
| Frame 3 | - | - | 1 | 1 | 1 | 1 | 0 | 0 | 0 | 3 | 3 | 3 | 3 |
| Status | F | F | F | F | H | F | F | F | F | F | F | H | H |

**Page Faults = 10**  
**Page Hits = 3**  
**Hit Ratio = 3/13 = 23.08%**

### 2. LRU (Least Recently Used)

| Page | 7 | 0 | 1 | 2 | 0 | 3 | 0 | 4 | 2 | 3 | 0 | 3 | 2 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Frame 1 | 7 | 7 | 7 | 2 | 2 | 2 | 2 | 4 | 4 | 4 | 0 | 0 | 0 |
| Frame 2 | - | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 3 | 3 | 3 | 3 |
| Frame 3 | - | - | 1 | 1 | 1 | 3 | 3 | 3 | 2 | 2 | 2 | 2 | 2 |
| Status | F | F | F | F | H | F | H | F | F | F | F | H | H |

**Page Faults = 9**  
**Page Hits = 4**  
**Hit Ratio = 4/13 = 30.77%**

### 3. Optimal Page Replacement

| Page | 7 | 0 | 1 | 2 | 0 | 3 | 0 | 4 | 2 | 3 | 0 | 3 | 2 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Frame 1 | 7 | 7 | 7 | 2 | 2 | 2 | 2 | 2 | 2 | 2 | 2 | 2 | 2 |
| Frame 2 | - | 0 | 0 | 0 | 0 | 0 | 0 | 4 | 4 | 4 | 0 | 0 | 0 |
| Frame 3 | - | - | 1 | 1 | 1 | 3 | 3 | 3 | 3 | 3 | 3 | 3 | 3 |
| Status | F | F | F | F | H | F | H | F | H | H | F | H | H |

**Page Faults = 7**  
**Page Hits = 6**  
**Hit Ratio = 6/13 = 46.15%**

*F = Page Fault, H = Page Hit*

### Comparison Table

| Algorithm | Page Faults | Page Hits | Hit Ratio |
|---|---:|---:|---:|
| FIFO | 10 | 3 | 23.08% |
| LRU | 9 | 4 | 30.77% |
| Optimal | 7 | 6 | 46.15% |

**Conclusion:** Optimal page replacement produces the fewest page faults for this reference string, followed by LRU and FIFO.



## Part C: Experiments and Analysis

### Experiment 1: Belady's Anomaly

**Reference String:** 1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5

#### Results with 3 Frames

| Algorithm | Page Faults | Page Hits |
|---|---:|---:|
| FIFO | 9 | 3 |
| LRU | 10 | 2 |
| Optimal | 7 | 5 |

#### Results with 4 Frames

| Algorithm | Page Faults | Page Hits |
|---|---:|---:|
| FIFO | 10 | 2 |
| LRU | 8 | 4 |
| Optimal | 6 | 6 |

**Q1. Does FIFO show Belady's anomaly?**

**Answer:** Yes. FIFO shows Belady's anomaly because its page faults increase from 9 with 3 frames to 10 with 4 frames. This means that increasing the number of frames can sometimes increase page faults in FIFO.

**Q2. Do LRU and Optimal show Belady's anomaly for this reference string?**

**Answer:** No. For this reference string, LRU page faults decrease from 10 to 8, and Optimal page faults decrease from 7 to 6 when the number of frames increases from 3 to 4.

### Experiment 2: Comparing FIFO and LRU

**Number of Frames:** 3

#### Case A: LRU Performs Better Than FIFO

**Reference String:** 4, 5, 3, 2, 3, 3, 3, 5, 4, 1, 5, 2, 1, 3, 1

| Algorithm | Page Faults | Page Hits |
|---|---:|---:|
| FIFO | 10 | 5 |
| LRU | 8 | 7 |

**Observation:** LRU performs better than FIFO because it produces 2 fewer page faults for this reference string.

#### Case B: FIFO Performs Better Than LRU

**Reference String:** 2, 5, 5, 3, 3, 4, 3, 5, 1, 2, 1, 4, 4, 2, 1

| Algorithm | Page Faults | Page Hits |
|---|---:|---:|
| FIFO | 6 | 9 |
| LRU | 7 | 8 |

**Observation:** FIFO performs better than LRU because it produces 1 fewer page fault for this reference string.

### Short Questions

**Q1. Why is Optimal Page Replacement not practically implementable in a real operating system?**

**Answer:** Optimal Page Replacement requires knowing which pages will be used in the future. Since a real operating system cannot predict future page references exactly, this algorithm cannot be implemented practically. It is mainly used as a benchmark.

**Q2. What extra hardware or bookkeeping does LRU need compared with FIFO?**

**Answer:** LRU needs to track when each page was last used. This requires extra bookkeeping, such as timestamps, counters, or a stack-like structure. FIFO mainly tracks the order in which pages entered memory.

**Q3. If the number of frames equals the number of distinct pages in the reference string, how many page faults will each algorithm produce, and why?**

**Answer:** FIFO, LRU, and Optimal will each produce a number of page faults equal to the number of distinct pages, assuming all frames are initially empty. Each distinct page causes one fault on its first reference, and no page needs to be replaced afterward.


